# RO-079 Behaviour trees and recovery: every number in the Note

The runs here are **our own Python version of Nav2's default behaviour tree** (`images/btkit.py`), not Nav2 running.
The control nodes copy the code of BehaviorTree.CPP v4 and of `nav2_behavior_tree` (branch `main`, read 2026-10-12);
the tree is `navigate_to_pose_w_replanning_and_recovery.xml`, ticked every 10 ms. The servers are stand-ins: A* on
0.1 m cells for the planner, a simple path follower with Nav2's goal and progress checkers for the controller, and
Spin, Wait and BackUp with the behaviour server's bring-up values. `images/apollokit.py` is our one-dimensional model
of Apollo's stop-sign scenario. Everything is deterministic (no random numbers).

In [1]:
import sys, math
sys.path.insert(0, 'images')
import numpy as np
import btkit as bt, bandkit as bk, navdraw as nd, apollokit as ak
from shapely.geometry import LineString, Point, Polygon, box
r2, r3 = (lambda x: round(float(x), 2)), (lambda x: round(float(x), 3))
R = bt.results()
def plen(P):
    return float(np.sum(np.hypot(*np.diff(P, axis=0).T)))
def events(run, uids):
    out, prev = [], {}
    for rec in run['records']:
        for lab, s, u in rec['log']:
            if u in uids and prev.get(u) != s:
                out.append((rec['t'], lab, s)); prev[u] = s
    return out
def recoveries(run):
    """(time, recovery) each time the round robin starts one of its four children."""
    out, was = [], set()
    for rec in run['records']:
        now = {u for _, s, u in rec['log'] if u in (31, 32, 33) and s == bt.RUNNING}
        for lab, s, u in rec['log']:
            if u == 28 or (u in now and u not in was and s == bt.RUNNING):
                out.append((rec['t'], lab))
        was = now
    return out

## 1. The rules of the control nodes on tiny trees (Sections 3 to 5)

In [2]:
ok, no = (lambda b: bt.SUCCESS), (lambda b: bt.FAILURE)
bb = dict(t=0.0, log=[])
print('Sequence [success, failure] ->', bt.Sequence('s', [bt.Leaf('a', ok), bt.Leaf('b', no)]).execute_tick(bb))
print('Fallback [failure, success] ->', bt.Fallback('f', [bt.Leaf('a', no), bt.Leaf('b', ok)]).execute_tick(bb))
print('Fallback [failure, failure] ->', bt.Fallback('f', [bt.Leaf('a', no), bt.Leaf('b', no)]).execute_tick(bb))
print('Inverter [failure] ->', bt.Inverter('i', [bt.Leaf('a', no)]).execute_tick(bb))
rr = bt.RoundRobin('r', [bt.Leaf('a', ok), bt.Leaf('b', ok), bt.Leaf('c', ok)])
got = []
for k in range(4):
    bb['log'] = []; rr.execute_tick(bb); got.append([l for l, s, u in bb['log'] if l != 'r'][0])
print('RoundRobin of three children that succeed, four ticks:', got)

Sequence [success, failure] -> FAILURE
Fallback [failure, success] -> SUCCESS
Fallback [failure, failure] -> FAILURE
Inverter [failure] -> SUCCESS
RoundRobin of three children that succeed, four ticks: ['a', 'b', 'c', 'a']


## 2. The finite state machine of Figure 2 (Section 2)

In [3]:
fsm = {'Plan': ['Drive', 'Clear', 'Spin', 'Wait', 'Back up', 'Aborted'], 'Drive': ['Plan', 'Arrived'],
       'Clear': ['Plan'], 'Spin': ['Plan'], 'Wait': ['Plan'], 'Back up': ['Plan']}
n = sum(len(v) for v in fsm.values())
states = ['Plan', 'Drive', 'Clear', 'Spin', 'Wait', 'Back up']
print('transitions:', n, '; states that can be left:', len(states), '; docking adds', len(states), 'in and 1 out:', n + len(states) + 1)

transitions: 12 ; states that can be left: 6 ; docking adds 6 in and 1 out: 19


## 3. The room, the robot and the plan (Section 1)

In [4]:
P0, _ = bt.astar(bt.START, bt.GOAL, dict(person=None, trolley=None))
print('A* path on 0.1 m cells, clear room:', r3(plen(P0)), 'm, under 4 m:', plen(P0) < 4.0)
for k in ('clear', 'person', 'blocked'):
    print(k, ':', R[k]['status'], 'at', R[k]['t'], 's; plans asked:', [(t, c) for t, c in R[k]['planned']][:3], '...')

A* path on 0.1 m cells, clear room: 3.38 m, under 4 m: True
clear : SUCCESS at 7.37 s; plans asked: [(0.01, None)] ...
person : SUCCESS at 9.77 s; plans asked: [(0.01, None), (3.02, None)] ...
blocked : SUCCESS at 15.6 s; plans asked: [(0.01, None), (3.02, 'NO_VALID_PATH'), (3.03, 'NO_VALID_PATH')] ...


## 4. The person run: the default tree replans only when the path is invalid (Sections 4 and 6)

In [5]:
P = R['person']
checks = [r['t'] for r in P['records'] if any(u == 11 for _, _, u in r['log'])]
print('1 Hz checks (ValidatePath ticked):', checks[:6])
print('answers:', [(r['t'], s) for r in P['records'] for l, s, u in r['log'] if u == 11][:5])
print('plans:', P['planned'])
print('noticed', r2(3.01 - bt.T_PERSON), 's after the person stepped in')
first = nd.at(P['records'], 0.01)['path']; second = nd.at(P['records'], 3.02)['path']
print('first path', r3(plen(first)), 'm; new path at 3.02 s', r3(plen(second)), 'm from the robot at', nd.at(P['records'], 3.02)['robot'][:2].round(2))
print('accepted at', P['t'], 's; clear room', R['clear']['t'], 's')
ticks = len(P['records'])
print('ticks:', ticks)

1 Hz checks (ValidatePath ticked): [1.01, 2.01, 3.01, 4.02, 5.02, 6.02]
answers: [(1.01, 'SUCCESS'), (2.01, 'SUCCESS'), (3.01, 'FAILURE'), (4.02, 'SUCCESS'), (5.02, 'SUCCESS')]
plans: [(0.01, None), (3.02, None)]
noticed 0.61 s after the person stepped in
first path 3.38 m; new path at 3.02 s 2.827 m from the robot at [2.55 2.08]
accepted at 9.77 s; clear room 7.37 s
ticks: 978


In [6]:
S, Q = R['mini_seq'], R['mini_pipe']
for name, run in (('Sequence', S), ('PipelineSequence', Q)):
    pv = [r['t'] for r in run['records'] if any(u == 2 for _, _, u in r['log'])]
    print(name, ': Path valid? ticked', len(pv), 'times, last at', pv[-1], '; plans', run['planned'])

Sequence : Path valid? ticked 1 times, last at 0.0 ; plans [(0.01, None)]
PipelineSequence : Path valid? ticked 309 times, last at 3.1 ; plans [(0.01, None), (2.41, None)]


## 5. The blocked door: recovery behaviours (Sections 1, 5 and 7)

In [7]:
Bk = R['blocked']
Pp = Point(*bt.GAP_PERSON).buffer(bk.PERSON_R, 128)
pairs = {'wall end to person': (LineString([(0, 4), (3.0, 4)]), Pp), 'person to table corner': (Pp, Polygon(bk.TABLE)),
         'trolley to table': (Polygon(bt.TROLLEY), Polygon(bk.TABLE)), 'trolley to cabinet': (Polygon(bt.TROLLEY), box(5.6, 3.4, 7.0, 4.0))}
for k, (a, b) in pairs.items():
    print(k, r3(a.distance(b)), 'm; robot width', 2 * bk.R_BODY, 'm')
print('person at', bt.GAP_PERSON, 'from', bt.T_PERSON, 's to', bt.T_LEAVES, 's; trolley', bt.TROLLEY.min(axis=0), 'to', bt.TROLLEY.max(axis=0))
print('robot at 3.01 s:', nd.at(Bk['records'], 3.01)['robot'].round(2))
print('planner answers:', Bk['planned'])
print('costmaps cleared:', Bk['cleared'])
print('recoveries started:', recoveries(Bk))
for e in events(Bk, {11, 12, 17, 21, 28, 31, 32, 33, 0}):
    print(e)

wall end to person 0.302 m; robot width 0.5 m
person to table corner 0.473 m; robot width 0.5 m
trolley to table 0.109 m; robot width 0.5 m
trolley to cabinet 0.1 m; robot width 0.5 m
person at [3.05 3.5 ] from 2.4 s to 8.0 s; trolley [5.35 2.65] to [5.75 3.3 ]
robot at 3.01 s: [2.55 2.07 1.35]
planner answers: [(0.01, None), (3.02, 'NO_VALID_PATH'), (3.03, 'NO_VALID_PATH'), (3.04, 'NO_VALID_PATH'), (3.05, 'NO_VALID_PATH'), (4.77, 'NO_VALID_PATH'), (4.78, 'NO_VALID_PATH'), (9.79, None)]
costmaps cleared: [(3.02, 'global'), (3.03, 'local'), (3.03, 'global'), (3.04, 'global'), (4.77, 'global')]
recoveries started: [(3.03, 'ClearingActions'), (3.05, 'Spin 1.57 rad'), (4.78, 'Wait 5 s')]
(0.0, 'ComputePathToPose', 'RUNNING')
(0.0, 'NavigateRecovery (retry 6)', 'RUNNING')
(0.01, 'ComputePathToPose', 'SUCCESS')
(0.01, 'FollowPath', 'RUNNING')
(1.01, 'ValidatePath', 'SUCCESS')
(3.01, 'ValidatePath', 'FAILURE')
(3.01, 'ComputePathToPose', 'RUNNING')
(3.02, 'ComputePathToPose', 'FAILURE')
(3.02

In [8]:
p_only, c1 = bt.astar(bt.START, bt.GOAL, dict(person=bt.GAP_PERSON, trolley=None))
both, c2 = bt.astar(bt.START, bt.GOAL, dict(person=bt.GAP_PERSON, trolley=bt.TROLLEY))
print('person only: path', r3(plen(p_only)), 'm (round the table); person and trolley:', c2)
sp = [r['t'] for r in Bk['records'] if any(u == 31 for _, _, u in r['log'])]
print('Spin from', sp[0], 'to', sp[-1], ':', r2(sp[-1] - sp[0]), 's; heading', r2(nd.at(Bk['records'], sp[0])['robot'][2]), '->', r2(nd.at(Bk['records'], sp[-1])['robot'][2]), 'rad')
wt = [r['t'] for r in Bk['records'] if any(u == 32 for _, _, u in r['log'])]
print('Wait from', wt[0], 'to', wt[-1], ':', r2(wt[-1] - wt[0]), 's')
print('robot drift during the spin (still slowing from 0.5 m/s at 0.5 m/s^2):', r3(np.hypot(*(nd.at(Bk['records'], sp[-1])['robot'][:2] - nd.at(Bk['records'], sp[0])['robot'][:2]))), 'm')

person only: path 8.645 m (round the table); person and trolley: NO_VALID_PATH
Spin from 3.05 to 4.76 : 1.71 s; heading 1.35 -> 2.95 rad
Wait from 4.78 to 9.78 : 5.0 s
robot drift during the spin (still slowing from 0.5 m/s at 0.5 m/s^2): 0.231 m


### Hand calculation: how long a 1.57 rad spin takes

The behaviour server commands `sqrt(2 x 3.2 x remaining angle)`, at most 1.0 rad/s and at least 0.4 rad/s, every 0.1 s.

In [9]:
a, wmax, wmin, total = 3.2, 1.0, 0.4, 1.57
r_full = wmax**2 / (2 * a)          # remaining angle below which the speed drops under 1.0 rad/s
r_min = wmin**2 / (2 * a)           # remaining angle below which the speed is held at 0.4 rad/s
t_full = (total - r_full) / wmax
t_slow = 2 * (math.sqrt(r_full) - math.sqrt(r_min)) / math.sqrt(2 * a)
t_last = r_min / wmin
print('r_full', r3(r_full), 'rad; r_min', r3(r_min), 'rad')
print('t_full', r3(t_full), 's; t_slow', r3(t_slow), 's; t_last', r3(t_last), 's; total', r3(t_full + t_slow + t_last), 's')

r_full 0.156 rad; r_min 0.025 rad
t_full 1.414 s; t_slow 0.188 s; t_last 0.063 s; total 1.664 s


## 6. The person stays: six recoveries, then failure (Section 7)

In [10]:
St = R['stays']
print('result', St['status'], 'at', St['t'], 's')
print('recoveries started:', recoveries(St))
for e in events(St, {21, 28, 31, 32, 33, 0}):
    print(e)
bu = [r for r in St['records'] if any(u == 33 for _, _, u in r['log'])]
print('BackUp moved', r3(np.hypot(*(bu[-1]['robot'][:2] - bu[0]['robot'][:2]))), 'm in', r2(bu[-1]['t'] - bu[0]['t']), 's')
print('recoveries that succeeded:', len([1 for r in St['records'] if (bt.SUCCESS, 21) in [(s, u) for _, s, u in r['log']]]))

result FAILURE at 13.58 s
recoveries started: [(3.03, 'ClearingActions'), (3.05, 'Spin 1.57 rad'), (4.78, 'Wait 5 s'), (9.8, 'BackUp 0.30 m'), (11.83, 'ClearingActions'), (11.85, 'Spin 1.57 rad')]
(0.0, 'NavigateRecovery (retry 6)', 'RUNNING')
(3.03, 'ClearingActions', 'SUCCESS')
(3.03, 'Recovery', 'SUCCESS')
(3.05, 'Spin 1.57 rad', 'RUNNING')
(3.05, 'Recovery', 'RUNNING')
(4.76, 'Spin 1.57 rad', 'SUCCESS')
(4.76, 'Recovery', 'SUCCESS')
(4.78, 'Wait 5 s', 'RUNNING')
(4.78, 'Recovery', 'RUNNING')
(9.78, 'Wait 5 s', 'SUCCESS')
(9.78, 'Recovery', 'SUCCESS')
(9.8, 'BackUp 0.30 m', 'RUNNING')
(9.8, 'Recovery', 'RUNNING')
(11.81, 'BackUp 0.30 m', 'SUCCESS')
(11.81, 'Recovery', 'SUCCESS')
(11.85, 'Spin 1.57 rad', 'RUNNING')
(11.85, 'Recovery', 'RUNNING')
(13.56, 'Spin 1.57 rad', 'SUCCESS')
(13.56, 'Recovery', 'SUCCESS')
(13.58, 'NavigateRecovery (retry 6)', 'FAILURE')
BackUp moved 0.3 m in 2.01 s
recoveries that succeeded: 6


## 7. The goal checker and the progress checker (Section 8)

In [11]:
B = Bk['records']
latch = min(r['t'] for r in B if r['checkers'] and r['checkers']['xy_reached'])
end = B[-1]
d_end = np.hypot(*(end['robot'][:2] - bt.GOAL[:2])); y_end = abs((bt.GOAL[2] - end['robot'][2] + math.pi) % (2 * math.pi) - math.pi)
rl = nd.at(B, latch)
print('within 0.25 m at', latch, 's:', r3(np.hypot(*(rl['robot'][:2] - bt.GOAL[:2]))), 'm, heading error', r3(abs((bt.GOAL[2] - rl['robot'][2] + math.pi) % (2 * math.pi) - math.pi)), 'rad')
print('accepted at', end['t'], 's:', r3(d_end), 'm and', r3(y_end), 'rad =', r2(math.degrees(y_end)), 'deg')
dmax = max(np.hypot(*(r['robot'][:2] - bt.GOAL[:2])) for r in B if r['t'] >= latch)
dmin = min(np.hypot(*(r['robot'][:2] - bt.GOAL[:2])) for r in B if r['t'] >= latch)
rp = rl['path']; k0 = int(np.argmin(np.hypot(*(rp - rl['robot'][:2]).T)))
print('path left at', latch, 's (the goal checker also needs under 1.0 m, path_length_tolerance):', r3(plen(rp[k0:])), 'm')
print('distance after the latch: largest', r3(dmax), 'm, smallest', r3(dmin), 'm')
Sk = R['stuck']['records']
fails = [r['t'] for r in Sk if any(l == 'FollowPath' and s == bt.FAILURE for l, s, u in r['log'])]
resets = sorted(set(r['checkers']['prog_t'] for r in Sk if r['checkers']))
print('wheel caught at', bt.STUCK_AT, 's; checkpoints at', resets, '; FollowPath fails at', fails)
r157 = nd.at(Sk, 1.57); r2s = nd.at(Sk, 2.0)
print('moved after the checkpoint at 1.57 s:', r3(np.hypot(*(r2s['robot'][:2] - r157['checkers']['prog_xy']))), 'm')
for e in events(R['stuck'], {16, 17, 18, 20, 21, 28}):
    print(e)

within 0.25 m at 14.6 s: 0.244 m, heading error 1.144 rad
accepted at 15.6 s: 0.171 m and 0.236 rad = 13.54 deg
path left at 14.6 s (the goal checker also needs under 1.0 m, path_length_tolerance): 0.271 m
distance after the latch: largest 0.244 m, smallest 0.14 m
wheel caught at 2.0 s; checkpoints at [0.01, 1.57, 11.62, 21.63] ; FollowPath fails at [11.62, 21.63]
moved after the checkpoint at 1.57 s: 0.214 m
(0.01, 'FollowPath', 'RUNNING')
(0.01, 'FollowPath (retry 1)', 'RUNNING')
(11.62, 'FollowPath', 'FAILURE')
(11.62, 'ClearLocalCostmap-Context', 'SUCCESS')
(11.62, 'ContextualControllerRecovery', 'SUCCESS')
(11.62, 'FollowPath', 'RUNNING')
(21.63, 'FollowPath', 'FAILURE')
(21.63, 'FollowPath (retry 1)', 'FAILURE')
(21.63, 'ClearingActions', 'SUCCESS')
(21.63, 'Recovery', 'SUCCESS')
(21.63, 'FollowPath', 'RUNNING')
(21.63, 'FollowPath (retry 1)', 'RUNNING')


## 8. Waypoint following (Section 9)

In [12]:
W = R['waypoints']
for w, r in zip(bt.WAYPOINTS, W['results']):
    print('waypoint', r['goal'] + 1, (w[0], w[1], round(math.degrees(w[2]))), ':', r['status'], 'sent at', r['t0'], 's, ended at', r['t'], 's')
print('plans for waypoint 1:', [t for t, c in W['planned'] if t < 10.47])
first = [r for r in W['records'] if r['goal'] == 0 and r['path'] is not None][0]
print('first path to waypoint 1:', r3(plen(first['path'])), 'm, over 4 m:', plen(first['path']) > 4.0)
print('recoveries started for waypoint 2:', [x for x in recoveries(W) if 10.4 < x[0] < 21.3])

waypoint 1 (6.0, 2.0, 90) : SUCCESS sent at 0.0 s, ended at 10.47 s
waypoint 2 (4.6, 3.5, 180) : FAILURE sent at 10.67 s, ended at 21.24 s
waypoint 3 (1.5, 2.0, 180) : SUCCESS sent at 21.24 s, ended at 34.41 s
plans for waypoint 1: [0.01, 1.02, 2.03, 3.04]
first path to waypoint 1: 5.267 m, over 4 m: True
recoveries started for waypoint 2: [(10.69, 'ClearingActions'), (10.71, 'Spin 1.57 rad'), (12.44, 'Wait 5 s'), (17.46, 'BackUp 0.30 m'), (19.49, 'ClearingActions'), (19.51, 'Spin 1.57 rad')]


## 9. Apollo's stop-sign scenario on our model (Section 10)

In [13]:
out = ak.simulate(26.0)
for e in ak.events(out):
    print(e)
cross_gone = [r['t'] for r in out if r['stage'] == 'STOP' and not r['watch']]
print('stopped at', [r['t'] for r in out if r['stage'] == 'STOP'][0], 's, front', r2(-[r['s'] for r in out if r['stage'] == 'STOP'][0]), 'm before the line')
print('crossing car leaves at', ak.T_GO, 's; its centre is 10.0 m from its line at', r2(ak.T_GO + math.sqrt(2 * (10.0 + (ak.CROSS_FRONT0 + ak.L_EGO / 2 - ak.CROSS_LINE)) / ak.A_CROSS)), 's')

(0.0, 'LANE_FOLLOW', None, '')
(5.5, 'STOP_SIGN_UNPROTECTED', 'PRE_STOP', 'front 4.0 m or less before the line')
(7.4, 'STOP_SIGN_UNPROTECTED', 'STOP', 'stopped 1.00 m before the line, at 0.08 m/s')
(13.2, 'STOP_SIGN_UNPROTECTED', 'CREEP', 'waited 5.8 s, watch list empty')
(17.0, 'STOP_SIGN_UNPROTECTED', 'INTERSECTION_CRUISE', 'past the creep point, 5 clear cycles')
(21.9, 'LANE_FOLLOW', None, 'rear 20 m past the line: scenario done')
stopped at 7.4 s, front 1.0 m before the line
crossing car leaves at 9.5 s; its centre is 10.0 m from its line at 13.17 s
